# Laboratorio 1 — Análisis espectral de la aceleración

Notebook adaptado a los tres archivos experimentales.

La DFT se aplica solamente a un intervalo aproximadamente periódico. Para identificar los armónicos se toma el pico fundamental dominante y se busca el segundo armónico en la frecuencia más cercana a 2 f1, respetando la definición física de armónico.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)

ARCHIVOS = {
    "10 Hz": "../datos/Biela-Manivela-10Hz-20s (negativo angulo).txt",
    "20 Hz": "../datos/Biela-Manivela-20Hz-20s.txt",
    "30 Hz": "../datos/Biela-Manivela-30Hz-20s.txt",
}

INTERVALOS = {
    "10 Hz": (5, 15),
    "20 Hz": (5, 15),
    "30 Hz": (8, 16),
}

COLUMNAS = ["t", "theta", "omega", "alpha", "x", "v", "a"]

def cargar_loggerpro(ruta):
    filas = []
    with open(ruta, "r", encoding="utf-8-sig") as f:
        lineas = f.readlines()[7:]
    for linea in lineas:
        partes = linea.rstrip("\n").split("\t")
        if len(partes) != 7:
            break
        try:
            filas.append([float(v) for v in partes])
        except ValueError:
            break
    return pd.DataFrame(filas, columns=COLUMNAS)

datos = {nombre: cargar_loggerpro(ruta) for nombre, ruta in ARCHIVOS.items()}


## 1. DFT de la aceleración

Se resta la media de la señal antes de calcular la FFT para reducir el componente DC. Se utiliza τ = t - t0 para que la fase obtenida por la FFT sea coherente con el tiempo de reconstrucción.


In [ ]:
resultados_fft = {}

for nombre, df in datos.items():
    inicio, fin = INTERVALOS[nombre]
    d = df[(df.t >= inicio) & (df.t <= fin)]

    t = d.t.to_numpy()
    tau = t - t[0]
    a_original = d.a.to_numpy()
    a = a_original - np.mean(a_original)

    N = len(a)
    dt = np.median(np.diff(tau))
    fs = 1/dt

    yf = np.fft.rfft(a)
    xf = np.fft.rfftfreq(N, dt)
    amplitudes = 2*np.abs(yf)/N
    amplitudes[0] /= 2

    # Primer armónico: pico dominante en el rango de la frecuencia fundamental.
    mask = (xf >= 0.3) & (xf <= 1.3)
    candidatos = np.where(mask)[0]
    i1 = candidatos[np.argmax(amplitudes[candidatos])]

    f1 = xf[i1]
    A1 = amplitudes[i1]
    phi1 = np.angle(yf[i1])

    # Segundo armónico: frecuencia más cercana a 2*f1.
    i2 = np.argmin(np.abs(xf - 2*f1))
    f2 = xf[i2]
    A2 = amplitudes[i2]
    phi2 = np.angle(yf[i2])

    reconstruida = (
        A1*np.cos(2*np.pi*f1*tau + phi1)
        + A2*np.cos(2*np.pi*f2*tau + phi2)
    )

    r2_recon = 1 - np.sum((a-reconstruida)**2)/np.sum((a-np.mean(a))**2)

    resultados_fft[nombre] = {
        "fs": fs, "N": N,
        "f1": f1, "A1": A1, "phi1": phi1,
        "f2": f2, "A2": A2, "phi2": phi2,
        "R2_reconstruccion": r2_recon
    }

    print(f"\n{nombre}: intervalo {inicio}–{fin} s | fs = {fs:.4f} Hz")
    print(f"1er armónico: f1 = {f1:.6f} Hz | A1 = {A1:.6f} | φ1 = {phi1:.6f} rad")
    print(f"2do armónico: f2 = {f2:.6f} Hz | A2 = {A2:.6f} | φ2 = {phi2:.6f} rad")
    print(f"R² reconstrucción con 2 armónicos = {r2_recon:.4f}")

    plt.figure()
    plt.stem(xf, amplitudes, basefmt=" ")
    plt.xlim(0, max(3.5, 2.5*f1))
    plt.xlabel("Frecuencia (Hz)")
    plt.ylabel("Amplitud")
    plt.title(f"Espectro de aceleración — {nombre}")
    plt.grid(True)
    plt.show()

    plt.figure()
    plt.plot(t, a_original, label="Original")
    plt.plot(t, reconstruida + np.mean(a_original), "--", label="1.º + 2.º armónico")
    plt.xlabel("Tiempo (s)")
    plt.ylabel("Aceleración (m/s²)")
    plt.title(f"Reconstrucción — {nombre}")
    plt.legend()
    plt.grid(True)
    plt.show()

tabla_fft = pd.DataFrame(resultados_fft).T
tabla_fft.index.name = "Muestreo"
display(tabla_fft)


## 2. Interpretación

Para el informe se debe:
- comparar f1 con la frecuencia del movimiento angular;
- verificar que f2 sea aproximadamente 2 f1;
- comparar A2 con A1;
- evaluar cuánto de la señal original conserva la reconstrucción;
- comentar armónicos adicionales y posibles causas experimentales.
